6번째 코드
==결과==
📊 Train Acc: 93.81% | Val Loss: 0.7836, Val Acc: 76.87%

🛑 5 Epoch 개선 없음. Early Stopping 발동!

1번 조건(neutral만 증강해서 학습): 오히려 Acc가 떨어짐.
--> 조건2 + 조건3: neutral에 loss 가중치를 높여 neutral:4감정 이진분류에서 neutral에 대한 중요도를 올려 attention과 비슷한 효과 ==> 7번코드에서 진행

In [1]:
# 환경 설정 및 라이브러리 임포트

import os
import json
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import random

# ================= 설정 =================
RAW_IMG_BASE = "/workspace/user1/dataset_v2/img/train" 
LBL_BASE = "/workspace/user1/dataset_v2/label_mod/train"
NPZ_BASE = "/workspace/user1/dataset_v2/segmentation/train"
AUG_IMG_BASE = "/workspace/user5/img_augmented/train"

SAVE_DIR = '/workspace/user1/model_v4_phase1'
os.makedirs(SAVE_DIR, exist_ok=True)

SEED = 42
BATCH_SIZE = 64 # RTX 3070 VRAM(8GB) 고려 안정적인 배치 사이즈
IMG_SIZE = 224
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

EMOTION_MAP = {
    'anger': 0, 'happy': 1, 'panic': 2, 'sadness': 3, 'neutral': 4,
    'blank': 4 
}
CLASS_NAMES = ['anger', 'happy', 'panic', 'sadness', 'neutral']

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"✅ 환경 설정 완료. 장치: {DEVICE}")

✅ 환경 설정 완료. 장치: cuda


In [2]:
# 데이터셋 정의 (Neutral 차별화 증강 및 오버샘플링 적용)

class Phase1EmotionDataset(Dataset):
    def __init__(self, raw_img_base, aug_img_base, lbl_base, base_transform=None, neutral_transform=None):
        self.base_transform = base_transform
        self.neutral_transform = neutral_transform
        self.samples = []
        
        neutral_samples = [] # 복제를 위한 임시 리스트

        print("🔍 원본 데이터(JSON) 로딩 및 크롭 준비 중...")
        if os.path.exists(lbl_base):
            for json_file in os.listdir(lbl_base):
                if not json_file.endswith('.json'): continue
                with open(os.path.join(lbl_base, json_file), 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    for item in data['images']:
                        if item.get('is_box_issue') == 'Yes': continue
                        
                        sample = {
                            'source': 'raw',
                            'img_path': os.path.join(raw_img_base, item['final_emotion'], item['filename']),
                            'bbox': item['bbox'],
                            'label': EMOTION_MAP[item['final_emotion']]
                        }
                        self.samples.append(sample)
                        if sample['label'] == 4: neutral_samples.append(sample)

        print("🔍 팀원 증강 데이터 로딩 중...")
        if os.path.exists(aug_img_base):
            for emotion_folder in EMOTION_MAP.keys():
                src_folder = os.path.join(aug_img_base, emotion_folder)
                if not os.path.isdir(src_folder): continue
                
                label = EMOTION_MAP[emotion_folder]
                file_list = [f for f in os.listdir(src_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
                
                for filename in file_list:
                    sample = {
                        'source': 'aug',
                        'img_path': os.path.join(src_folder, filename),
                        'label': label
                    }
                    self.samples.append(sample)
                    if label == 4: neutral_samples.append(sample)
        
        # [핵심] Neutral 데이터 5배 오버샘플링
        self.samples.extend(neutral_samples * 4) 
        print(f"✅ 로딩 완료. (Neutral 추가 확보 후 총 데이터 수: {len(self.samples)}장)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = cv2.imread(item['img_path'])
        
        if image is None: return self.__getitem__(random.randint(0, len(self)-1))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if item['source'] == 'raw':
            box = item['bbox']
            x1, y1, x2, y2 = max(0, box['minX']), max(0, box['minY']), box['maxX'], box['maxY']
            if x2 <= x1 or y2 <= y1: return self.__getitem__(random.randint(0, len(self)-1))
            image = image[y1:y2, x1:x2]
            
        # [핵심] Neutral 전용 기하학적 증강 추가 적용
        if item['label'] == 4 and self.neutral_transform:
            image = self.neutral_transform(image=image)['image']
            
        if self.base_transform:
            image = self.base_transform(image=image)['image']
            
        return image, torch.tensor(item['label'], dtype=torch.long)

print("✅ Phase1EmotionDataset 클래스 정의 완료.")

✅ Phase1EmotionDataset 클래스 정의 완료.


In [3]:
# Transform 및 DataLoader 선언

# 1. Neutral 전용 기하학적 증강 (얼굴 각도, 윤곽의 미세한 왜곡으로 과적합 방지)
neutral_specific_transform = A.Compose([
    A.OneOf([
        A.Perspective(scale=(0.05, 0.1), p=0.5), # 카메라 각도 변화 시뮬레이션
        A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.5), # 미세한 표정 변화 생성
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5)
    ], p=0.8)
])

# 2. 공통 Base Transform (너무 무겁지 않게 조절)
base_train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

train_dataset = Phase1EmotionDataset(
    RAW_IMG_BASE, AUG_IMG_BASE, LBL_BASE, 
    base_transform=base_train_transform, 
    neutral_transform=neutral_specific_transform
)

# Val 데이터셋 설정 (원본 기준)
VAL_IMG_BASE = "/workspace/user1/dataset_v2/img/val"
VAL_LBL_BASE = "/workspace/user1/dataset_v2/label_mod/val"

class ValRawDataset(Dataset):
    def __init__(self, img_base, lbl_base, transform=None):
        self.transform = transform
        self.samples = []
        if os.path.exists(lbl_base):
            for json_file in os.listdir(lbl_base):
                if not json_file.endswith('.json'): continue
                with open(os.path.join(lbl_base, json_file), 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    for item in data['images']:
                        if item.get('is_box_issue') == 'Yes': continue
                        self.samples.append({
                            'img_path': os.path.join(img_base, item['final_emotion'], item['filename']),
                            'bbox': item['bbox'],
                            'label': EMOTION_MAP[item['final_emotion']]
                        })

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        item = self.samples[idx]
        image = cv2.imread(item['img_path'])
        if image is None: return self.__getitem__(0)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        box = item['bbox']
        image = image[max(0, box['minY']):box['maxY'], max(0, box['minX']):box['maxX']]
        if self.transform: image = self.transform(image=image)['image']
        return image, torch.tensor(item['label'], dtype=torch.long)

val_dataset = ValRawDataset(VAL_IMG_BASE, VAL_LBL_BASE, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"⚡ DataLoader 설정 완료.")

/home/user1/.conda/envs/mwkim/lib/python3.10/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


🔍 원본 데이터(JSON) 로딩 및 크롭 준비 중...
🔍 팀원 증강 데이터 로딩 중...
✅ 로딩 완료. (Neutral 추가 확보 후 총 데이터 수: 48230장)
⚡ DataLoader 설정 완료.


In [4]:
# 모델 로드 및 조기 종료(Early Stopping) 셋팅
def get_model(num_classes=5):
    model = timm.create_model('efficientnet_b2', pretrained=True, num_classes=num_classes)
    fer_url = "https://github.com/sb-ai-lab/EmotiEffLib/raw/main/models/affectnet_emotions/enet_b2_8.pt"
    
    print("🚀 FER 전용 가중치 주입 중...")
    try:
        loaded = torch.hub.load_state_dict_from_url(fer_url, map_location='cpu', weights_only=False)
        if hasattr(loaded, 'state_dict'): loaded = loaded.state_dict()
        filtered_dict = {k: v for k, v in loaded.items() if 'classifier' not in k and 'head' not in k and 'fc' not in k}
        model.load_state_dict(filtered_dict, strict=False)
        print("✅ 가중치 주입 성공!")
    except Exception as e:
        print(f"⚠️ 주입 실패. 기본 가중치 사용. ({e})")
    return model

model = get_model(num_classes=5).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

# 🚨 수정: verbose=True 인자를 삭제했습니다.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

print("✅ 학습 설정 완료.")

🚀 FER 전용 가중치 주입 중...
✅ 가중치 주입 성공!
✅ 학습 설정 완료.


In [5]:
# 과적합 방지 학습 루프 실행

NUM_EPOCHS = 30
PATIENCE = 5 # 5 에포크 동안 개선 없으면 즉시 종료
best_val_loss = float('inf')
trigger_times = 0

SAVE_PATH = os.path.join(SAVE_DIR, 'best_phase1_model.pth')
scaler = torch.amp.GradScaler('cuda')

print("🚀 [Phase 1] 차별화 증강 기반 학습 시작...")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    train_pbar = tqdm(train_loader, desc=f"Ep [{epoch+1}/{NUM_EPOCHS}] Train")
    for images, labels in train_pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    train_loss = running_loss / len(train_dataset)
    train_acc = correct_train / total_train * 100

    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_dataset)
    val_acc = correct_val / total_val * 100
    
    # 스케줄러 업데이트
    scheduler.step(val_loss)

    print(f"📊 Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  💾 저장됨 (Val Loss: {val_loss:.4f})")
        trigger_times = 0
    else:
        trigger_times += 1
        if trigger_times >= PATIENCE:
            print(f"\n🛑 {PATIENCE} Epoch 개선 없음. Early Stopping 발동!")
            break

print(f"\n✅ 학습 완료. 최고 모델: {SAVE_PATH}")

🚀 [Phase 1] 차별화 증강 기반 학습 시작...


Ep [1/30] Train:   0%|          | 0/754 [00:00<?, ?it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [1/30] Train:   1%|          | 9/754 [00:21<12:49,  1.03s/it, loss=1.7800]  Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [1/30] Train:   5%|▌         | 41/754 [01:03<08:51,  1.34it/s, loss=1.4257]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [1/30] Train:   8%|▊         | 57/754 [01:23<08:47,  1.32it/s, loss=1.3054]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [1/30] Train:  24%|██▍       | 180/754 [04:01<11:46,  1.23s/it, loss=1.8142]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [1/30] Train:  25%|██▍       | 185/754 [04:06<08:22,  1.13it/s, loss=1.3673]Invalid SOS parameters for sequenti

📊 Train Acc: 61.52% | Val Loss: 2.6757, Val Acc: 69.39%
  💾 저장됨 (Val Loss: 2.6757)


Ep [2/30] Train:   1%|          | 4/754 [00:05<12:30,  1.00s/it, loss=0.8609]  Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [2/30] Train:  10%|▉         | 74/754 [01:30<08:44,  1.30it/s, loss=0.8063]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [2/30] Train:  24%|██▍       | 182/754 [03:45<08:03,  1.18it/s, loss=0.7904]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [2/30] Train:  28%|██▊       | 214/754 [04:24<07:51,  1.15it/s, loss=0.8437]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [2/30] Train:  29%|██▉       | 222/754 [04:34<08:08,  1.09it/s, loss=0.8761]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [2/30] Train:  34%|███▍      | 256/754 [05:18<10:07,  1.22s/it, loss=0.6813]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [2/30] T

📊 Train Acc: 76.04% | Val Loss: 0.7354, Val Acc: 77.04%
  💾 저장됨 (Val Loss: 0.7354)


Ep [3/30] Train:   0%|          | 0/754 [00:00<?, ?it/s]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [3/30] Train:   1%|          | 4/754 [00:05<13:27,  1.08s/it, loss=0.5997]  Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [3/30] Train:   1%|          | 8/754 [00:10<09:47,  1.27it/s, loss=0.5713]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [3/30] Train:  10%|▉         | 73/754 [01:32<09:35,  1.18it/s, loss=0.5659]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [3/30] Train:  16%|█▌        | 117/754 [02:24<07:54,  1.34it/s, loss=0.6769]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [3/30] Train:  25%|██▌       | 189/754 [03:54<07:15,  1.30it/s, loss=0.6829]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [3/30] Train:  28%|██▊       | 20

... (5 outputs left out)

Ep [6/30] Train:   0%|          | 3/754 [00:05<17:00,  1.36s/it, loss=0.3723]  Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [6/30] Train:   1%|          | 8/754 [00:11<09:34,  1.30it/s, loss=0.3230]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [6/30] Train:   2%|▏         | 16/754 [00:20<08:28,  1.45it/s, loss=0.5497]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [6/30] Train:   7%|▋         | 55/754 [01:11<11:03,  1.05it/s, loss=0.4469]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [6/30] Train:   8%|▊         | 59/754 [01:16<11:44,  1.01s/it, loss=0.3993]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [6/30] Train:   9%|▉         | 67/754 [01:26<11:23,  1.00it/s, loss=0.4148]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [6/30] Train:

📊 Train Acc: 93.21% | Val Loss: 0.7644, Val Acc: 77.47%


Ep [7/30] Train:   1%|          | 8/754 [00:11<11:51,  1.05it/s, loss=0.3869]  Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [7/30] Train:   3%|▎         | 22/754 [00:28<12:36,  1.03s/it, loss=0.3116]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [7/30] Train:  13%|█▎        | 100/754 [02:04<08:26,  1.29it/s, loss=0.4292]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [7/30] Train:  19%|█▊        | 140/754 [02:53<07:14,  1.41it/s, loss=0.3197]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [7/30] Train:  23%|██▎       | 172/754 [03:32<06:36,  1.47it/s, loss=0.3307]Invalid SOS parameters for sequential JPEG
Invalid SOS parameters for sequential JPEG
Ep [7/30] Train:  25%|██▍       | 188/754 [03:52<07:08,  1.32it/s, loss=0.3468]Invalid SOS parameters for sequential JPEG
Invalid SOS

📊 Train Acc: 93.81% | Val Loss: 0.7836, Val Acc: 76.87%

🛑 5 Epoch 개선 없음. Early Stopping 발동!

✅ 학습 완료. 최고 모델: /workspace/user1/model_v4_phase1/best_phase1_model.pth
